설계행렬 A는 절편 열과 특성 열을 포함한다. 제곱오차 합을 최소화하는 조건은 정규방정식 `AᵀA w = Aᵀy`다. 역행렬 표현에는 가역성 조건이 필요하다. 실제 계산은 역행렬을 명시적으로 만드는 대신 **A에 직접 최소제곱 풀이를 적용**한다.

오차가 독립이고 같은 분산 σ²을 갖는 평균 0의 정규분포라는 모형에서, 고정된 양의 σ에 대한 로그우도는 상수항과 제곱오차 항으로 나뉜다. 따라서 w에 대한 최대우도는 최소제곱과 연결된다. **실제 시계열 오차가 이 가정을 만족한다는 증명은 아니다.** CLT만으로 오차 가정을 정당화하지 않는다.


In [1]:
import numpy as np

In [27]:
def fit_least_squares(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    b = np.ones([X.shape[0],1])
    A = np.concatenate([b,X], axis=1)
    u, s, vt = np.linalg.svd(A,full_matrices=False)
    c = u.T@y
    z = np.where(s != 0, c/s, 0)
    beta = vt.T@z
    return beta
# 1. 정확한 직선 → 기대 계수: [1, 2]
X = np.array([[0], [1], [2]])
print(X.shape)
y = np.array([1, 3, 5])
print(fit_least_squares(X,y))
# 2. 음수 기울기 → 기대 계수: [3, -2]
X = np.array([[-2], [0], [2]])
y = np.array([7, 3, -1])
print(fit_least_squares(X,y))
# 3. 잡음 포함 → 기대 계수: 약 [0.666667, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 2, 5])
print(fit_least_squares(X,y))

(3, 1)
[1. 2.]
[ 3. -2.]
[0.66666667 2.        ]


In [10]:
def gaussian_log_likelihood(w: np.ndarray, X: np.ndarray, y: np.ndarray, sigma: float = 1.0) -> float:
    #검증
    if X.ndim != 2:
        raise ValueError
    if w.ndim != 1:
        raise ValueError
    if y.ndim != 1:
        raise ValueError
    if X.shape[0] != y.shape[0]:
        raise ValueError
    if w.shape[0] != X.shape[1]+1:
        raise ValueError
    if not (np.all(np.isfinite(X)) and np.all(np.isfinite(y)) and np.all(np.isfinite(w)) and np.all(np.isfinite(sigma))):
        raise ValueError
    if sigma <= 0:
        raise ValueError
    A = np.concatenate([np.ones([X.shape[0],1]),X], axis=1)
    l = -X.shape[0]*np.log(np.sqrt(2*np.pi)*sigma)-1/(2*sigma**2)*np.sum((y-A@w)**2)
    return l
# 1. 정확한 직선 → 기대 계수: [1, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 3, 5])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))
# 2. 음수 기울기 → 기대 계수: [3, -2]
X = np.array([[-2], [0], [2]])
y = np.array([7, 3, -1])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))
# 3. 잡음 포함 → 기대 계수: 약 [0.666667, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 2, 5])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))

# 
X = np.array([[0], [1]])
y = np.array([1, 3])
print(gaussian_log_likelihood(np.array([1,2]),X,y))

-2.756815599614018
-2.756815599614018
-3.0901489329473515
-1.8378770664093453


In [42]:
# 1. x축 데이터 생성: 0부터 10까지 5개의 점 생성
x = np.linspace(0, 10, 1000)
np.random.shuffle(x)
x=x.reshape(1000,1)
# 2. 직선 방정식 설정 (기울기 m=2, y절편 b=3)
m = 2.1
b = 3
y = m * x + b
y = y.reshape(1000,)
print(fit_least_squares(x,y))
gaussian_log_likelihood(fit_least_squares(x,y),x,y)



[3.  2.1]


np.float64(-918.9385332046727)

In [50]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
b = np.ones((1000,1))
a = np.concatenate([b,x],axis=1)
lin.fit(x,y)
print(lin.coef_[0], lin.intercept_)
use = np.array([lin.intercept_, lin.coef_[0]])
gaussian_log_likelihood(use,x,y)

2.0999999999999988 3.0000000000000036


np.float64(-918.9385332046727)